# Álgebra con NumPy — datos de la Premier League 2023/24

In [1]:
import numpy as np
import csv
from datetime import datetime

with open('data/E0_2324.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    goles = []
    equipos = []
    arbitros = []
    amarillas = []
    rojas = []
    fechas = []
    cuotas = []
    for row in reader:
        resultado = []
        resultado.append(row['FTHG'])
        resultado.append(row['FTAG'])
        goles.append(resultado)
        equipo = []
        equipo.append(row['HomeTeam'])
        equipo.append(row['AwayTeam'])
        equipos.append(equipo)
        
        arbitros.append(row['Referee'])

        amarillas.append(int(row['HY']) + int(row['AY']) )
        
        rojas.append(int(row['HR']) + int(row['AR']))

        fechas.append(row['Date'])

        cuota = []
        cuota.append(row['B365H'])
        cuota.append(row['B365D'])
        cuota.append(row['B365A'])
        cuotas.append(cuota)

    g = np.array(goles, dtype=np.int64)
    e = np.array(equipos)
    a = np.array(arbitros)
    r_card = np.array(rojas, dtype=np.int64)
    y_card = np.array(amarillas, dtype=np.int64)
    f = np.array(fechas)
    c = np.array(cuotas, dtype=np.float32)

## 1. ¿Cuántos partidos terminaron 0-0?
Forma esperada del resultado: empates tiene de forma ()

In [2]:
empates = ((g[:,0] == 0) & (g[:,1] == 0)).sum()
print(f'Hubo {empates} partidos con resultado 0-0')

Hubo 11 partidos con resultado 0-0


Hubieron 11 partidos en la temporada con marcador de 0-0

## 2. ¿Cuál fue el resultado (marcador exacto) que más se repitió en la temporada?

In [3]:
repetido, conteo = np.unique(g, axis=0, return_counts=True)
indice_max = np.argmax(conteo)
partido_repetido = repetido[indice_max]
veces_repetidas = conteo[indice_max]

print(f'El marcador mas repetido fue {partido_repetido[0]}-{partido_repetido[1]} con un total de {veces_repetidas} repeticiones ')

El marcador mas repetido fue 1-1 con un total de 38 repeticiones 


El marcador mas repetido en la temporada fue 1-1 con un total de 38 repeticiones 

## 3. ¿Qué equipo marcó más goles jugando de visitante?
Forma esperada del resultado: equipos unicos = (20,) , goles visitante = () , mejor_visitante = (20,) , indice_max_goles = ()

In [4]:
equipos_unicos = np.unique(e)
lista_goles_visitante = []
for i in equipos_unicos:
    mascara_visitante = (e[:,1] == i)
    goles_visitante = g[:,1].sum(where = mascara_visitante)
    lista_goles_visitante.append(goles_visitante)
mejor_visitante = np.array(lista_goles_visitante)
indice_max_goles = np.argmax(mejor_visitante)
print(f'{equipos_unicos[indice_max_goles]}: {mejor_visitante[indice_max_goles]}')

print(equipos_unicos.shape)
print(goles_visitante.shape)
print(mejor_visitante.shape)
print(indice_max_goles.shape)


Man City: 45
(20,)
()
(20,)
()


El equipo que más goles marcó jugando de visitante fue el manchester city, con un total de 45 tantos

## 4. ¿Qué equipo encajó menos goles en toda la temporada?

In [5]:
lista_goles_contra = []
for i in equipos_unicos:
    mascara_local = (e[:,0] == i)
    goles_local_contra = g[:, 1].sum(where=mascara_local) 

    mascara_visitante = (e[:,1] == i)
    goles_visitante_contra = g[:, 0].sum(where=mascara_visitante) 

    total_goles_contra = g[:, 1].sum(where=mascara_local) 
    lista_goles_contra.append(total_goles_contra)

goles_contra = np.array(lista_goles_contra)
indice_min_goles_contra = np.argmin(goles_contra)
print(f'{equipos_unicos[indice_min_goles_contra]}: {goles_contra[indice_min_goles_contra]}')

Arsenal: 16


El equipo con menos goles encajados en toda la temporada fue el Arsenal, solo recibio 29 goles.

## 5. ¿Cuál fue el partido con mayor diferencia de goles? ¿Hubo empates en esa diferencia máxima?

In [6]:
diferencias = np.abs(g[:, 0] - g[:, 1])
max_diferencia = np.max(diferencias)
indices_max = np.where(diferencias == max_diferencia)[0]

print(f"La mayor diferencia de goles en la temporada fue de: {max_diferencia} goles.")
print(f"¿Hubo empates (varios partidos con esta diferencia)? {'Sí' if len(indices_max) > 1 else 'No'}")
print(f"Cantidad de partidos con esta diferencia máxima: {len(indices_max)}\n")

print("Detalle del/los partido(s):")
for idx in indices_max:
    equipo_local = e[idx, 0]
    equipo_visitante = e[idx, 1]
    goles_local = g[idx, 0]
    goles_visitante = g[idx, 1]
    print(f"- {equipo_local} {goles_local} - {goles_visitante} {equipo_visitante}")

La mayor diferencia de goles en la temporada fue de: 8 goles.
¿Hubo empates (varios partidos con esta diferencia)? No
Cantidad de partidos con esta diferencia máxima: 1

Detalle del/los partido(s):
- Sheffield United 0 - 8 Newcastle


El partido con mayor diferencia de goles fue el partido entre el Sheffield United y Newcastle con un marcador de 0 - 8 donde el Newcastle salio victorioso.

## 6. ¿Qué árbitro pitó más partidos?

In [7]:
arbitro_unico, conteo_a = np.unique(a, axis=0, return_counts=True)
indice_arbi_max = np.argmax(conteo_a)
arbi_repetido = arbitro_unico[indice_arbi_max]
veces_repetidas_arbi = conteo_a[indice_arbi_max]

print(f'El arbitro que mas partidos arbitró fue {arbi_repetido} con un total de {veces_repetidas_arbi} partidos ')

El arbitro que mas partidos arbitró fue A Taylor con un total de 27 partidos 


El arbitro que mas partidos arbitró fue A Taylor con un total de 27 partidos 

## 7. ¿Qué árbitro mostró más tarjetas por partido?
Criterio usado (solo amarillas / amarillas y rojas): amarillas y rojas
Forma esperada del resultado: r_card = (380,) , y_card = (380,), amarillas_arbitro = (), rojas_arbitro = (), arbi_tarjetas = (29,) , max_tarjetas = (), tarjetas_partido = (29,), mas_tarjetas_partido = ()

In [8]:
tarjetas_arbitro = []
for i in arbitro_unico:
    mascara_arbitros = (a[:] == i)
    amarillas_arbitro = y_card[:].sum(where=mascara_arbitros)
    rojas_arbitro = r_card[:].sum(where=mascara_arbitros)
    total_tarjetas = rojas_arbitro + amarillas_arbitro
    tarjetas_arbitro.append(total_tarjetas)

arbi_tarjetas = np.array(tarjetas_arbitro)
max_tarjetas = np.argmax(arbi_tarjetas)

ratio_tarjetas = []
for x in range( len(arbitro_unico)):
    if conteo_a[x] > 10:
        ratio =  tarjetas_arbitro[x] / conteo_a[x]
        ratio_tarjetas.append(ratio)
    else:
        ratio_tarjetas.append(0)
tarjetas_partido = np.array(ratio_tarjetas)
mas_tarjetas_partido = np.argmax(tarjetas_partido) 

print(f' {arbitro_unico[mas_tarjetas_partido]} : {tarjetas_partido[mas_tarjetas_partido]:.2f}, que sacó {arbi_tarjetas[mas_tarjetas_partido]} tarjetas en {conteo_a[mas_tarjetas_partido]} partidos pitados')
print(f' {arbitro_unico[max_tarjetas]} : {tarjetas_partido[max_tarjetas]:.2f}, que sacó {arbi_tarjetas[max_tarjetas]} tarjetas en {conteo_a[max_tarjetas]} partidos pitados')

print(r_card.shape)
print(y_card.shape)
print(amarillas_arbitro.shape)
print(rojas_arbitro.shape)
print(arbi_tarjetas.shape)
print(max_tarjetas.shape)
print(tarjetas_partido.shape)
print(mas_tarjetas_partido.shape)


 D Coote : 5.31, que sacó 85 tarjetas en 16 partidos pitados
 A Taylor : 4.85, que sacó 131 tarjetas en 27 partidos pitados
(380,)
(380,)
()
()
(29,)
()
(29,)
()


El arbitro que sacó más tarjetas por partido fue D Coote, con 85 tarjetas sacadas en 16 partidos pitados, con un ratio de 5.31 tarjetas por partido. Sin embargo el arbitro que más tarjetas sacó fue A Taylor : 131, aunque el lo hizo en 27 partidos, con un promedio de 4.85 tarjetas por partido. Excluyo a los árbitros con 10 partidos o menos porque con muestras pequeñas el promedio es inestable

## 8. ¿En qué mes se marcaron más goles?

In [9]:
goles_totales_partido = g[:, 0] + g[:, 1]

meses = np.array([f[3:5] for f in fechas])

meses_unicos = np.unique(meses)

lista_goles_mes = []

for mes in meses_unicos:
  mascara_mes = meses == mes
  goles_mes = goles_totales_partido[mascara_mes].sum()
  lista_goles_mes.append(goles_mes)

array_goles_meses = np.array(lista_goles_mes)
indice_max = np.argmax(array_goles_meses)

print(
    f"El mes con más goles fue: {meses_unicos[indice_max]} con"
    f" {array_goles_meses[indice_max]} goles."
)


El mes con más goles fue: 12 con 206 goles.


Se marcaron mas goles en diciembre, con un total de 206 goles

## 9. ¿Cuál fue la racha más larga de victorias consecutivas del Manchester City?
Definición: partidos del City (de local o de visitante), en orden cronológico,
contando victorias seguidas.

In [10]:
jugo_city = (e[:, 0] == 'Man City') | (e[:, 1] == 'Man City')
equipos_city = e[jugo_city]  
goles_city = g[jugo_city]  

es_local = equipos_city[:, 0] == 'Man City'
gano_local = es_local & (goles_city[:, 0] > goles_city[:, 1])
es_visitante = equipos_city[:, 1] == 'Man City'
gano_visitante = es_visitante & (goles_city[:, 1] > goles_city[:, 0])

victorias_city = gano_local | gano_visitante

racha_actual = 0
racha_maxima = 0

for gano in victorias_city:
  if gano:
    racha_actual += 1
    if racha_actual > racha_maxima:
      racha_maxima = racha_actual
  else:
    racha_actual = 0

print(
    f"La racha más larga de victorias consecutivas del Manchester City fue:"
    f" {racha_maxima}"
)




La racha más larga de victorias consecutivas del Manchester City fue: 9


La racha más larga de victorias consecutivas del Manchester City fue de 9

## 10. ¿Acertaron las casas de apuestas? ¿Qué porcentaje de partidos ganó el favorito?
El favorito es el equipo con la cuota más baja entre B365H, B365D y B365A.
Este porcentaje es la línea base que tendré que superar en el proyecto P1.

In [11]:
import numpy as np


favorito = np.argmin(c, axis=1)


resultado_real = np.empty(len(g), dtype=int)
resultado_real[g[:, 0] > g[:, 1]] = 0  
resultado_real[g[:, 0] == g[:, 1]] = 1  
resultado_real[g[:, 0] < g[:, 1]] = 2  

aciertos = (favorito == resultado_real)

total_partidos = len(c)
partidos_acertados = aciertos.sum()
porcentaje = (partidos_acertados / total_partidos) * 100

print(f"Partidos en los que ganó el favorito: {partidos_acertados} de {total_partidos}")
print(f"Porcentaje de acierto de las casas de apuestas: {porcentaje:.2f}%")

Partidos en los que ganó el favorito: 227 de 380
Porcentaje de acierto de las casas de apuestas: 59.74%


El favorito ganó 227 de 380 partidos, dandole un 59.74% de aceierto a las casas de apuesta